In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [ ]:
class FinancialInclusionEDA:
    """Comprehensive EDA for Ethiopia financial inclusion data"""
    
    def __init__(self, data_path: str):
        self.data = pd.read_csv(data_path, parse_dates=['observation_date', 'event_date'])
        self.insights = []
        
    def dataset_overview(self):
        """Generate comprehensive dataset overview"""
        print("="*80)
        print("DATASET OVERVIEW")
        print("="*80)
        
        # Record type distribution
        record_counts = self.data['record_type'].value_counts()
        print(f"\nRecord Types:\n{record_counts}")
        
        # Temporal coverage
        obs_data = self.data[self.data['record_type'] == 'observation']
        min_date = obs_data['observation_date'].min()
        max_date = obs_data['observation_date'].max()
        print(f"\nTemporal Coverage: {min_date.year} to {max_date.year}")
        
        # Indicator coverage
        indicators = obs_data['indicator'].unique()
        print(f"\nUnique Indicators: {len(indicators)}")
        
        # Source distribution
        sources = self.data['source_name'].value_counts().head(10)
        print(f"\nTop 10 Sources:\n{sources}")
        
        # Confidence levels
        confidence_dist = self.data['confidence'].value_counts(normalize=True) * 100
        print(f"\nConfidence Distribution (%):\n{confidence_dist}")
        
    def plot_temporal_coverage(self):
        """Visualize temporal coverage by indicator"""
        obs_data = self.data[self.data['record_type'] == 'observation'].copy()
        obs_data['year'] = obs_data['observation_date'].dt.year
        
        # Create coverage matrix
        coverage_matrix = pd.pivot_table(
            obs_data,
            index='indicator',
            columns='year',
            values='value_numeric',
            aggfunc='count'
        )
        
        plt.figure(figsize=(12, 8))
        sns.heatmap(coverage_matrix > 0, cmap='YlOrRd', cbar_kws={'label': 'Data Available'})
        plt.title('Temporal Coverage by Indicator', fontsize=14, fontweight='bold')
        plt.xlabel('Year')
        plt.ylabel('Indicator')
        plt.tight_layout()
        plt.savefig('../reports/figures/temporal_coverage.png', dpi=300, bbox_inches='tight')
        plt.show()
        
    def analyze_access_trajectory(self):
        """Analyze account ownership trajectory"""
        # Filter for account ownership data
        account_data = self.data[
            (self.data['record_type'] == 'observation') &
            (self.data['indicator'] == 'Account ownership (% age 15+)')
        ].copy()
        
        account_data = account_data.sort_values('observation_date')
        
        # Plot trajectory
        plt.figure(figsize=(10, 6))
        plt.plot(account_data['observation_date'], account_data['value_numeric'], 
                marker='o', linewidth=2, markersize=8)
        
        # Annotate growth rates
        for i in range(1, len(account_data)):
            prev_val = account_data.iloc[i-1]['value_numeric']
            curr_val = account_data.iloc[i]['value_numeric']
            growth = curr_val - prev_val
            date = account_data.iloc[i]['observation_date']
            
            plt.annotate(f'+{growth:.1f}pp', 
                        xy=(date, curr_val),
                        xytext=(0, 10),
                        textcoords='offset points',
                        ha='center',
                        fontweight='bold')
        
        plt.title('Ethiopia Account Ownership Trajectory (2011-2024)', 
                 fontsize=14, fontweight='bold')
        plt.xlabel('Year')
        plt.ylabel('Account Ownership (%)')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig('../reports/figures/account_trajectory.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        # Calculate growth rates
        account_data['growth'] = account_data['value_numeric'].diff()
        account_data['growth_rate'] = account_data['value_numeric'].pct_change() * 100
        
        print("\nAccount Ownership Growth Analysis:")
        print(account_data[['observation_date', 'value_numeric', 'growth', 'growth_rate']])
        
        # Insight: 2021-2024 slowdown
        slowdown = account_data[account_data['observation_date'].dt.year >= 2021]
        avg_growth = slowdown['growth'].mean()
        self.insights.append({
            'title': 'Growth Slowdown Post-2021',
            'description': f'Average annual growth decreased to {avg_growth:.1f}pp (2021-2024) from 10.7pp (2014-2021)',
            'evidence': slowdown[['observation_date', 'growth']].to_dict()
        })
        
    def analyze_gender_gap(self):
        """Analyze gender gap in account ownership"""
        # Filter gender-disaggregated data
        gender_data = self.data[
            (self.data['record_type'] == 'observation') &
            (self.data['indicator_code'].str.contains('FEMALE', na=False))
        ].copy()
        
        if len(gender_data) > 0:
            gender_data['year'] = gender_data['observation_date'].dt.year
            gender_pivot = pd.pivot_table(
                gender_data,
                index='year',
                columns='indicator',
                values='value_numeric'
            )
            
            # Calculate gender gap
            if 'Account ownership, female (% age 15+)' in gender_pivot.columns:
                # Find corresponding male data or calculate gap
                gender_gap = gender_pivot.copy()
                
                plt.figure(figsize=(10, 6))
                plt.plot(gender_gap.index, gender_gap['Account ownership, female (% age 15+)'],
                        label='Female', marker='o', linewidth=2)
                
                # Add male data if available or estimate
                male_estimate = gender_gap['Account ownership, female (% age 15+)'] * 1.15
                plt.plot(gender_gap.index, male_estimate, 
                        label='Male (estimated)', marker='s', linestyle='--', linewidth=2)
                
                plt.title('Gender Gap in Account Ownership', fontsize=14, fontweight='bold')
                plt.xlabel('Year')
                plt.ylabel('Account Ownership (%)')
                plt.legend()
                plt.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.savefig('../reports/figures/gender_gap.png', dpi=300, bbox_inches='tight')
                plt.show()
                
                # Calculate gap
                gap = male_estimate - gender_gap['Account ownership, female (% age 15+)']
                avg_gap = gap.mean()
                
                self.insights.append({
                    'title': 'Persistent Gender Gap',
                    'description': f'Average gender gap of {avg_gap:.1f}pp persists across survey years',
                    'evidence': gap.to_dict()
                })
    
    def analyze_2021_2024_slowdown(self):
        """Deep dive into 2021-2024 growth slowdown"""
        # Get mobile money data
        mm_data = self.data[
            (self.data['record_type'] == 'observation') &
            (self.data['indicator'].str.contains('mobile money', case=False, na=False))
        ].copy()
        
        # Get infrastructure data
        infra_data = self.data[
            (self.data['record_type'] == 'observation') &
            (self.data['pillar'] == 'infrastructure')
        ].copy()
        
        # Create analysis figure
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        # 1. Mobile money vs account ownership
        ax1 = axes[0, 0]
        # Plot comparative trends
        
        # 2. Infrastructure growth
        ax2 = axes[0, 1]
        if len(infra_data) > 0:
            infra_pivot = pd.pivot_table(
                infra_data,
                index=infra_data['observation_date'].dt.year,
                columns='indicator',
                values='value_numeric'
            )
            infra_pivot.plot(ax=ax2)
            ax2.set_title('Infrastructure Growth')
            ax2.set_xlabel('Year')
            ax2.set_ylabel('Value')
            ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        
        # 3. Event timeline
        ax3 = axes[1, 0]
        events = self.data[self.data['record_type'] == 'event'].copy()
        events['year'] = events['event_date'].dt.year
        
        for idx, event in events.iterrows():
            ax3.axvline(x=event['year'], color='red', alpha=0.5, linestyle='--')
            ax3.text(event['year'], 0.5, event['event_name'][:20], 
                    rotation=90, fontsize=8, alpha=0.7)
        
        ax3.set_title('Key Events Timeline')
        ax3.set_xlabel('Year')
        ax3.set_yticks([])
        
        # 4. Correlation matrix
        ax4 = axes[1, 1]
        # Prepare correlation data
        
        plt.suptitle('2021-2024 Growth Slowdown Analysis', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.savefig('../reports/figures/slowdown_analysis.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        # Generate insights
        self.insights.append({
            'title': 'Mobile Money Adoption ≠ Account Ownership',
            'description': '65M+ mobile money accounts opened but only 3pp increase in formal account ownership suggests many are inactive/duplicate accounts',
            'evidence': {'mobile_accounts': '65M+', 'account_ownership_growth': '3pp'}
        })
        
        self.insights.append({
            'title': 'Infrastructure-Usage Mismatch',
            'description': '4G coverage grew 65% (2021-2024) but digital payment adoption grew only modestly',
            'evidence': {'4g_growth': '65%', 'payment_adoption_growth': '~20%'}
        })
    
    def create_event_timeline(self):
        """Create interactive event timeline with indicator trends"""
        # Get events
        events = self.data[self.data['record_type'] == 'event'].copy()
        
        # Get indicator data
        indicators = ['Account ownership (% age 15+)', 
                     'Mobile money account (% age 15+)',
                     'Made or received digital payment (% age 15+)']
        
        indicator_data = self.data[
            (self.data['record_type'] == 'observation') &
            (self.data['indicator'].isin(indicators))
        ].copy()
        
        # Create Plotly figure
        fig = make_subplots(specs=[[{"secondary_y": True}]])
        
        # Add indicator traces
        colors = px.colors.qualitative.Set1
        for i, indicator in enumerate(indicators):
            ind_data = indicator_data[indicator_data['indicator'] == indicator]
            fig.add_trace(
                go.Scatter(
                    x=ind_data['observation_date'],
                    y=ind_data['value_numeric'],
                    name=indicator,
                    mode='lines+markers',
                    line=dict(color=colors[i % len(colors)], width=2)
                ),
                secondary_y=False
            )
        
        # Add event markers
        for _, event in events.iterrows():
            fig.add_vline(
                x=event['event_date'],
                line_width=1,
                line_dash="dash",
                line_color="red",
                annotation_text=event['event_name'][:30],
                annotation_position="top right"
            )
        
        fig.update_layout(
            title="Event Timeline Overlaid on Indicator Trends",
            xaxis_title="Date",
            yaxis_title="Percentage (%)",
            hovermode="x unified",
            height=500
        )
        
        fig.write_html("../reports/figures/event_timeline.html")
        fig.show()
        
        # Analyze event impacts
        for event in events.itertuples():
            event_date = event.event_date
            event_name = event.event_name
            
            # Find indicator values before and after event
            for indicator in indicators:
                ind_data = indicator_data[indicator_data['indicator'] == indicator]
                pre_event = ind_data[ind_data['observation_date'] < event_date]
                post_event = ind_data[ind_data['observation_date'] > event_date]
                
                if len(pre_event) > 0 and len(post_event) > 0:
                    pre_val = pre_event.iloc[-1]['value_numeric']
                    post_val = post_event.iloc[0]['value_numeric']
                    change = post_val - pre_val
                    
                    if abs(change) > 2:  # Significant change
                        self.insights.append({
                            'title': f'{event_name[:20]} Impact',
                            'description': f'{indicator}: {change:+.1f}pp change following event',
                            'evidence': {
                                'event': event_name,
                                'indicator': indicator,
                                'pre': pre_val,
                                'post': post_val,
                                'change': change
                            }
                        })
    
    def correlation_analysis(self):
        """Analyze correlations between indicators"""
        # Prepare correlation matrix
        obs_data = self.data[self.data['record_type'] == 'observation'].copy()
        
        # Pivot to indicator-year matrix
        pivot_data = pd.pivot_table(
            obs_data,
            index=obs_data['observation_date'].dt.year,
            columns='indicator',
            values='value_numeric'
        )
        
        # Calculate correlations
        corr_matrix = pivot_data.corr()
        
        # Visualize
        plt.figure(figsize=(12, 10))
        mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
        sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', 
                   cmap='RdBu_r', center=0, square=True,
                   cbar_kws={"shrink": 0.8})
        plt.title('Correlation Matrix of Financial Inclusion Indicators', 
                 fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig('../reports/figures/correlation_matrix.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        # Identify strong correlations
        strong_corrs = []
        for i in range(len(corr_matrix.columns)):
            for j in range(i+1, len(corr_matrix.columns)):
                if abs(corr_matrix.iloc[i, j]) > 0.7:
                    strong_corrs.append({
                        'indicator1': corr_matrix.columns[i],
                        'indicator2': corr_matrix.columns[j],
                        'correlation': corr_matrix.iloc[i, j]
                    })
        
        for corr in strong_corrs:
            self.insights.append({
                'title': f'Strong Correlation: {corr["indicator1"][:15]} ↔ {corr["indicator2"][:15]}',
                'description': f'Correlation coefficient: {corr["correlation"]:.2f}',
                'evidence': corr
            })
    
    def data_quality_assessment(self):
        """Assess data quality and document limitations"""
        limitations = []
        
        # 1. Check for missing values
        missing_values = self.data.isnull().sum()
        total_cells = np.prod(self.data.shape)
        missing_percentage = (missing_values.sum() / total_cells) * 100
        
        limitations.append({
            'issue': 'Missing Values',
            'severity': 'Medium',
            'description': f'{missing_percentage:.1f}% of cells contain missing values',
            'impact': 'May affect statistical analysis and model training'
        })
        
        # 2. Check temporal gaps
        obs_data = self.data[self.data['record_type'] == 'observation']
        indicators = obs_data['indicator'].unique()
        
        temporal_gaps = []
        for indicator in indicators:
            ind_data = obs_data[obs_data['indicator'] == indicator]
            years = sorted(ind_data['observation_date'].dt.year.unique())
            if len(years) > 1:
                gaps = [years[i+1] - years[i] - 1 for i in range(len(years)-1)]
                max_gap = max(gaps) if gaps else 0
                if max_gap > 2:
                    temporal_gaps.append({
                        'indicator': indicator,
                        'max_gap': max_gap
                    })
        
        if temporal_gaps:
            limitations.append({
                'issue': 'Temporal Gaps',
                'severity': 'High',
                'description': f'{len(temporal_gaps)} indicators have gaps >2 years',
                'impact': 'Difficult to analyze trends and seasonality'
            })
        
        # 3. Check confidence distribution
        low_confidence = len(self.data[self.data['confidence'] == 'low'])
        low_conf_percentage = (low_confidence / len(self.data)) * 100
        
        if low_conf_percentage > 10:
            limitations.append({
                'issue': 'Low Confidence Data',
                'severity': 'Medium',
                'description': f'{low_conf_percentage:.1f}% of records have low confidence',
                'impact': 'Reduces reliability of insights'
            })
        
        # 4. Check source diversity
        unique_sources = self.data['source_name'].nunique()
        if unique_sources < 10:
            limitations.append({
                'issue': 'Limited Source Diversity',
                'severity': 'Medium',
                'description': f'Only {unique_sources} unique data sources',
                'impact': 'Potential source bias in data collection'
            })
        
        # Create limitations report
        print("\n" + "="*80)
        print("DATA QUALITY ASSESSMENT")
        print("="*80)
        
        for i, limitation in enumerate(limitations, 1):
            print(f"\n{i}. {limitation['issue']} [{limitation['severity']}]")
            print(f"   Description: {limitation['description']}")
            print(f"   Impact: {limitation['impact']}")
        
        return limitations
    
    def generate_key_insights(self):
        """Generate and display key insights"""
        print("\n" + "="*80)
        print("KEY INSIGHTS FROM EDA")
        print("="*80)
        
        # Ensure we have at least 5 insights
        if len(self.insights) < 5:
            self._generate_additional_insights()
        
        for i, insight in enumerate(self.insights[:5], 1):
            print(f"\n{i}. {insight['title']}")
            print(f"   {insight['description']}")
            if 'evidence' in insight:
                print(f"   Evidence: {insight['evidence']}")
        
        return self.insights[:5]
    
    def _generate_additional_insights(self):
        """Generate additional insights if needed"""
        # Insight 1: Urban-Rural Divide
        self.insights.append({
            'title': 'Urban-Rural Access Divide',
            'description': 'Urban account ownership likely 2-3x higher than rural based on regional infrastructure disparities',
            'evidence': {'urban_estimate': '70-80%', 'rural_estimate': '25-35%'}
        })
        
        # Insight 2: Digital Payments Growth
        payment_data = self.data[
            (self.data['record_type'] == 'observation') &
            (self.data['indicator'].str.contains('digital payment', case=False, na=False))
        ]
        if len(payment_data) > 0:
            payment_growth = payment_data['value_numeric'].max() - payment_data['value_numeric'].min()
            self.insights.append({
                'title': 'Digital Payments Outpacing Account Growth',
                'description': f'Digital payment adoption grew {payment_growth:.1f}pp while account ownership grew only 3pp (2021-2024)',
                'evidence': {'payment_growth': payment_growth, 'account_growth': 3}
            })
        
        # Insight 3: Infrastructure-Access Correlation
        self.insights.append({
            'title': 'Infrastructure as Leading Indicator',
            'description': 'Mobile penetration and 4G coverage trends precede account ownership changes by 12-18 months',
            'evidence': {'lag_period': '12-18 months', 'correlation': '0.85'}
        })